# Stage 04 · Release And Signoff

本 notebook 独立运行，只消费 stage package，并调用 scripts/04_Release_And_Signoff.py。

### 初始化运行常量与通用工具
显式配置 drive_project_root、stage 01 输入包、可选 stage 02 / stage 03 输入包，以及默认配置文件与脚本入口。
- 仓库：https://github.com/RICHAAARC/CEG-WM.git
- 分支：organize

In [ ]:
# pyright
# - reportMissingImports=false
# - reportUnknownParameterType=false
# - reportMissingParameterType=false
# - reportUnknownArgumentType=false
# - reportUnknownVariableType=false
# - reportUnknownMemberType=false
# - reportUnknownLambdaType=false
from pathlib import Path
from typing import Any, cast
import json
import os
import shutil
import subprocess
import sys

NOTEBOOK_NAME = "04_Release_And_Signoff"
REPO_URL = "https://github.com/RICHAAARC/CEG-WM.git"
REPO_BRANCH = "organize"
REPO_ROOT = Path("/content/ceg_wm_workspace")
DRIVE_MOUNT_ROOT = Path("/content/drive")
DRIVE_PROJECT_ROOT = DRIVE_MOUNT_ROOT / "MyDrive" / "CEG_WM_Outputs_project_root"
CONFIG_PATH = REPO_ROOT / "configs" / "default.yaml"
SCRIPT_PATH = REPO_ROOT / "scripts" / "04_Release_And_Signoff.py"

STAGE_01_PACKAGE_PATH = None
STAGE_02_PACKAGE_PATH = None
STAGE_03_PACKAGE_PATH = None
REQUIRE_STAGE_02 = True
REQUIRE_STAGE_03 = True

def run_checked(command: list[str], cwd: Path | None = None, env: dict[str, str] | None = None) -> None:
    print("$", " ".join(str(part) for part in command))
    subprocess.run(command, cwd=cwd, env=env, check=True)

def ensure_dir(path: Path) -> Path:
    path.mkdir(parents=True, exist_ok=True)
    return path

def load_json(path: Path) -> dict[str, Any]:
    with path.open("r", encoding="utf-8") as handle:
        return json.load(handle)

def print_json(title: str, payload: Any) -> None:
    print(f"\n[{title}]")
    print(json.dumps(payload, ensure_ascii=False, indent=2, sort_keys=True))

### 挂载 Google Drive 并同步代码仓库
挂载 Google Drive、准备输出根目录，并克隆或更新代码仓库，确保 signoff 审计基于当前仓库真实代码。

In [ ]:
from google.colab import drive  # type: ignore[import-not-found]

drive_module = cast(Any, drive)
drive_module.mount(str(DRIVE_MOUNT_ROOT), force_remount=True)
ensure_dir(DRIVE_PROJECT_ROOT)

if REPO_ROOT.exists() and (REPO_ROOT / ".git").exists():
    origin_url = subprocess.run(
        ["git", "-C", str(REPO_ROOT), "remote", "get-url", "origin"],
        check=False,
        capture_output=True,
        text=True,
        encoding="utf-8",
        errors="replace",
    ).stdout.strip()
    if origin_url.rstrip("/") != REPO_URL.rstrip("/"):
        shutil.rmtree(REPO_ROOT)
        run_checked(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO_ROOT)])
    else:
        run_checked(["git", "fetch", "origin"], cwd=REPO_ROOT)
        run_checked(["git", "checkout", REPO_BRANCH], cwd=REPO_ROOT)
        run_checked(["git", "pull", "origin", REPO_BRANCH], cwd=REPO_ROOT)
else:
    if REPO_ROOT.exists():
        shutil.rmtree(REPO_ROOT)
    run_checked(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO_ROOT)])

print_json(
    "repo_binding",
    {
        "repo_url": REPO_URL,
        "repo_branch": REPO_BRANCH,
        "repo_root": str(REPO_ROOT),
        "config_path": str(CONFIG_PATH),
        "script_path": str(SCRIPT_PATH),
        "drive_project_root": str(DRIVE_PROJECT_ROOT),
    },
)

### 安装依赖并准备最小运行环境
stage 04 不重跑主链，不需要 GPU 或模型下载；这里只安装仓库本身和最小依赖，用于 package 审计与 signoff。

In [ ]:
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

if Path("/usr/bin/apt-get").exists():
    subprocess.run(["apt-get", "update", "-qq"], check=False, capture_output=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", "git", "unzip"], check=False, capture_output=True)

run_checked([sys.executable, "-m", "pip", "install", "--upgrade", "pip", "setuptools", "wheel"], cwd=REPO_ROOT)
if (REPO_ROOT / "pyproject.toml").exists():
    run_checked([sys.executable, "-m", "pip", "install", "-e", str(REPO_ROOT)], cwd=REPO_ROOT)
else:
    run_checked([sys.executable, "-m", "pip", "install", "-r", str(REPO_ROOT / "requirements.txt")], cwd=REPO_ROOT)

print_json(
    "stage_role_binding",
    {
        "stage_name": NOTEBOOK_NAME,
        "consumes_only_stage_packages": True,
        "rerun_main_mechanism": False,
        "default_config_path": str(CONFIG_PATH),
        "stage_01_package_path": STAGE_01_PACKAGE_PATH or "<auto_discover_latest_formal_package>",
        "stage_02_package_path": STAGE_02_PACKAGE_PATH or "<auto_discover_if_available>",
        "stage_03_package_path": STAGE_03_PACKAGE_PATH or "<auto_discover_if_available>",
        "require_stage_02": REQUIRE_STAGE_02,
        "require_stage_03": REQUIRE_STAGE_03,
    },
)

### 环境预检与输入包门禁
在执行 signoff 脚本前，先解析手工输入或自动发现的 stage package，并检查配置文件、脚本文件，以及各 stage package 的 formal 合法性与 required / optional 约束。

In [ ]:
from scripts.notebook_runtime_common import resolve_stage_package_input_or_discover

PRECHECK_RESULTS: list[dict[str, Any]] = []

def record_precheck(name: str, ok: bool, detail: str) -> None:
    PRECHECK_RESULTS.append({
        "name": str(name),
        "ok": bool(ok),
        "detail": str(detail),
    })

STAGE_PACKAGE_CONFIGS: dict[str, dict[str, Any]] = {
    "stage_01": {
        "stage_name": "01_Paper_Full_Cuda",
        "manual_input": STAGE_01_PACKAGE_PATH,
        "export_stage_root": DRIVE_PROJECT_ROOT / "exports" / "01_Paper_Full_Cuda",
        "required": True,
    },
    "stage_02": {
        "stage_name": "02_Parallel_Attestation_Statistics",
        "manual_input": STAGE_02_PACKAGE_PATH,
        "export_stage_root": DRIVE_PROJECT_ROOT / "exports" / "02_Parallel_Attestation_Statistics",
        "required": bool(REQUIRE_STAGE_02),
    },
    "stage_03": {
        "stage_name": "03_Experiment_Matrix_Full",
        "manual_input": STAGE_03_PACKAGE_PATH,
        "export_stage_root": DRIVE_PROJECT_ROOT / "exports" / "03_Experiment_Matrix_Full",
        "required": bool(REQUIRE_STAGE_03),
    },
}
PRECHECK_STAGE_PACKAGE_RESOLUTIONS: dict[str, dict[str, Any]] = {}
PRECHECK_STAGE_PACKAGE_SUMMARY: dict[str, dict[str, Any]] = {}

def _resolved_package_path(resolution: dict[str, Any]) -> Path | None:
    selected_package_path = resolution.get("selected_package_path")
    if not bool(resolution.get("selected_package_valid", resolution.get("formal_package_valid", False))):
        return None
    if isinstance(selected_package_path, str) and selected_package_path not in {"", "<absent>"}:
        return Path(selected_package_path)
    return None

record_precheck("配置文件存在", CONFIG_PATH.exists(), str(CONFIG_PATH))
record_precheck("主流程脚本存在", SCRIPT_PATH.exists(), str(SCRIPT_PATH))

for module_name in ["yaml"]:
    try:
        __import__(module_name)
        record_precheck(f"模块可导入: {module_name}", True, "ok")
    except Exception as exc:
        record_precheck(f"模块可导入: {module_name}", False, f"{type(exc).__name__}: {exc}")

for stage_key, stage_cfg in STAGE_PACKAGE_CONFIGS.items():
    resolution = resolve_stage_package_input_or_discover(
        stage_cfg["manual_input"],
        stage_cfg["export_stage_root"],
        expected_stage_name=str(stage_cfg["stage_name"]),
    )
    PRECHECK_STAGE_PACKAGE_RESOLUTIONS[stage_key] = resolution
    stage_summary = {
        "stage_name": str(stage_cfg["stage_name"]),
        "required": bool(stage_cfg["required"]),
        "requirement": "required" if stage_cfg["required"] else "optional",
        "manual_input_used": bool(resolution.get("manual_input_used", False)),
        "manual_input_path": resolution.get("manual_input_path", "<absent>"),
        "auto_discovered_package_path": resolution.get("auto_discovered_package_path", "<absent>"),
        "selected_package_path": resolution.get("selected_package_path", "<absent>"),
        "package_exists": bool(resolution.get("package_exists", False)),
        "formal_package_valid": bool(resolution.get("selected_package_valid", resolution.get("formal_package_valid", False))),
        "package_role": resolution.get("package_role", "<absent>"),
        "package_discovery_scope": resolution.get("package_discovery_scope", "<absent>"),
        "package_policy_source": resolution.get("package_policy_source", "absent"),
        "manifest_parseable": bool(resolution.get("manifest_parseable", False)),
        "package_index_parseable": bool(resolution.get("package_index_parseable", False)),
        "stage_manifest_stage_name": resolution.get("stage_manifest_stage_name", "<absent>"),
        "manifest_stage_name": resolution.get("manifest_stage_name", "<absent>"),
        "stage_name_matches": bool(resolution.get("stage_name_matches", False)),
        "diagnostics_like": bool(resolution.get("diagnostics_like", False)),
        "resolution_error": resolution.get("resolution_error"),
    }
    PRECHECK_STAGE_PACKAGE_SUMMARY[stage_key] = stage_summary
    record_precheck(
        f"{stage_cfg['stage_name']} package 解析",
        stage_summary["formal_package_valid"] or not stage_summary["required"],
        json.dumps(stage_summary, ensure_ascii=False, sort_keys=True),
    )

RESOLVED_STAGE_PACKAGE_INPUTS = PRECHECK_STAGE_PACKAGE_RESOLUTIONS
RESOLVED_STAGE_PACKAGE_SUMMARY = PRECHECK_STAGE_PACKAGE_SUMMARY
RESOLVED_STAGE_01_PACKAGE_PATH = _resolved_package_path(PRECHECK_STAGE_PACKAGE_RESOLUTIONS["stage_01"])
RESOLVED_STAGE_02_PACKAGE_PATH = _resolved_package_path(PRECHECK_STAGE_PACKAGE_RESOLUTIONS["stage_02"])
RESOLVED_STAGE_03_PACKAGE_PATH = _resolved_package_path(PRECHECK_STAGE_PACKAGE_RESOLUTIONS["stage_03"])

hard_fail = []
for stage_key, stage_cfg in STAGE_PACKAGE_CONFIGS.items():
    stage_summary = PRECHECK_STAGE_PACKAGE_SUMMARY[stage_key]
    if stage_summary["manual_input_used"] and not stage_summary["formal_package_valid"]:
        hard_fail.append({
            "name": f"{stage_cfg['stage_name']} manual package invalid",
            "detail": json.dumps(stage_summary, ensure_ascii=False, sort_keys=True),
        })
        continue
    if stage_summary["required"] and not stage_summary["formal_package_valid"]:
        hard_fail.append({
            "name": f"{stage_cfg['stage_name']} package required",
            "detail": json.dumps(stage_summary, ensure_ascii=False, sort_keys=True),
        })

print_json(
    "environment_precheck",
    {
        "items": PRECHECK_RESULTS,
        "resolved_stage_packages": PRECHECK_STAGE_PACKAGE_SUMMARY,
        "hard_fail": hard_fail,
    },
)
if hard_fail:
    raise RuntimeError(f"environment precheck failed: {hard_fail}")

### 执行 stage 04 signoff 脚本并读取结果
复用上一格 precheck 已解析的 stage package 路径，调用 scripts/04_Release_And_Signoff.py，只做 package 审计、lineage 校验、release_manifest 与 signoff_report 生成，然后读取本次 stage 04 summary。

In [ ]:
from scripts.notebook_runtime_common import build_repo_import_subprocess_env, make_stage_run_id

if "RESOLVED_STAGE_01_PACKAGE_PATH" not in globals():
    raise RuntimeError("resolved stage packages are missing; run the precheck cell first")
if RESOLVED_STAGE_01_PACKAGE_PATH is None:
    raise RuntimeError("stage 01 package is unresolved; fix the precheck failures before executing stage 04")

STAGE_RUN_ID = make_stage_run_id(NOTEBOOK_NAME)
COMMAND = [
    sys.executable,
    str(SCRIPT_PATH),
    "--drive-project-root",
    str(DRIVE_PROJECT_ROOT),
    "--stage-01-package",
    str(RESOLVED_STAGE_01_PACKAGE_PATH),
    "--config",
    str(CONFIG_PATH),
    "--notebook-name",
    NOTEBOOK_NAME,
    "--stage-run-id",
    STAGE_RUN_ID,
    "--require-stage-02",
    str(REQUIRE_STAGE_02).lower(),
    "--require-stage-03",
    str(REQUIRE_STAGE_03).lower(),
]
if RESOLVED_STAGE_02_PACKAGE_PATH is not None:
    COMMAND.extend(["--stage-02-package", str(RESOLVED_STAGE_02_PACKAGE_PATH)])
if RESOLVED_STAGE_03_PACKAGE_PATH is not None:
    COMMAND.extend(["--stage-03-package", str(RESOLVED_STAGE_03_PACKAGE_PATH)])

print_json(
    "resolved_command_inputs",
    {
        "stage_01_package_path": str(RESOLVED_STAGE_01_PACKAGE_PATH),
        "stage_02_package_path": (
            str(RESOLVED_STAGE_02_PACKAGE_PATH) if RESOLVED_STAGE_02_PACKAGE_PATH is not None else "<absent>"
        ),
        "stage_03_package_path": (
            str(RESOLVED_STAGE_03_PACKAGE_PATH) if RESOLVED_STAGE_03_PACKAGE_PATH is not None else "<absent>"
        ),
        "require_stage_02": REQUIRE_STAGE_02,
        "require_stage_03": REQUIRE_STAGE_03,
    },
)
NOTEBOOK_SUBPROCESS_ENV = build_repo_import_subprocess_env(repo_root=REPO_ROOT)
run_checked(COMMAND, cwd=REPO_ROOT, env=NOTEBOOK_SUBPROCESS_ENV)

SUMMARY_PATH = DRIVE_PROJECT_ROOT / "runtime_state" / NOTEBOOK_NAME / STAGE_RUN_ID / "stage_summary.json"
STAGE_SUMMARY = load_json(SUMMARY_PATH)
STAGE_MANIFEST = load_json(Path(STAGE_SUMMARY["stage_manifest_path"]))
PACKAGE_MANIFEST = load_json(Path(STAGE_SUMMARY["package_manifest_path"]))
SIGNOFF_REPORT = load_json(Path(STAGE_SUMMARY["signoff_report_path"]))
RELEASE_MANIFEST = load_json(Path(STAGE_SUMMARY["release_manifest_path"]))

REQUIRED_STAGE_04_FILES = {
    "stage_summary": SUMMARY_PATH,
    "stage_manifest": Path(STAGE_SUMMARY["stage_manifest_path"]),
    "package_manifest": Path(STAGE_SUMMARY["package_manifest_path"]),
    "signoff_report": Path(STAGE_SUMMARY["signoff_report_path"]),
    "release_manifest": Path(STAGE_SUMMARY["release_manifest_path"]),
    "stage_package": Path(STAGE_SUMMARY["package_path"]),
}
missing_outputs = [name for name, path_obj in REQUIRED_STAGE_04_FILES.items() if not path_obj.exists()]
if missing_outputs:
    raise RuntimeError(f"stage 04 required outputs missing: {missing_outputs}")

print_json("stage_summary", STAGE_SUMMARY)
print_json("stage_manifest", STAGE_MANIFEST)
print_json("package_manifest", PACKAGE_MANIFEST)
print_json("signoff_report", SIGNOFF_REPORT)
print_json("release_manifest", RELEASE_MANIFEST)
print(f"signoff_report_path: {STAGE_SUMMARY['signoff_report_path']}")
print(f"stage_04_package_path: {STAGE_SUMMARY['package_path']}")
print(f"stage_04_package_sha256: {STAGE_SUMMARY['package_sha256']}")

### 最终保存在 Google Drive 中的文件路径
本 notebook 的输入和输出都位于 Google Drive 挂载目录下。stage 04 只读取前序 stage package，并写出自己的 signoff 工件与 package。

```text
/content/drive/MyDrive/CEG_WM_Outputs_project_root/
├─ runtime_state/
│  └─ 04_Release_And_Signoff/
│     └─ <stage_run_id>/
│        ├─ stage_summary.json
│        ├─ runtime_metadata/runtime_config_snapshot.yaml
│        └─ lineage/
│           ├─ source_stage_01_manifest.json
│           ├─ source_stage_01_package_manifest.json
│           ├─ source_stage_02_manifest.json         [若提供]
│           ├─ source_stage_02_package_manifest.json [若提供]
│           ├─ source_stage_03_manifest.json         [若提供]
│           └─ source_stage_03_package_manifest.json [若提供]
├─ <run_root，对应 stage_summary.json 中的 run_root>/
│  └─ artifacts/
│     ├─ stage_manifest.json
│     ├─ workflow_summary.json
│     ├─ run_closure.json
│     ├─ signoff/signoff_report.json
│     └─ release/release_manifest.json
├─ <export_root，对应 stage_summary.json 中的 export_root>/
│  ├─ 04_Release_And_Signoff__<stage_run_id>.zip
│  ├─ package_manifest.json
│  └─ package_index.json
└─ <package_path，对应 stage_summary.json 中的 package_path>
   └─ stage 04 最终 zip 包
```

最终 zip 包只包含 stage 04 自己的 signoff 与 release 结果，以及前序 stage 的 manifest 级 lineage 快照；不会把 01、02、03 的业务产物整包复制进来。zip 内部的核心相对路径如下：

```text
artifacts/stage_manifest.json
artifacts/workflow_summary.json
artifacts/run_closure.json
artifacts/signoff/signoff_report.json
artifacts/release/release_manifest.json
artifacts/package_manifest.json
artifacts/package_index.json
runtime_metadata/runtime_config_snapshot.yaml
lineage/source_stage_01_manifest.json
lineage/source_stage_01_package_manifest.json
lineage/source_stage_02_manifest.json                 [若提供]
lineage/source_stage_02_package_manifest.json         [若提供]
lineage/source_stage_03_manifest.json                 [若提供]
lineage/source_stage_03_package_manifest.json         [若提供]
```

其中：
1. 本阶段新增结果是 signoff_report、release_manifest、workflow_summary、run_closure。
2. 来自 01、02、03 的内容只保留 manifest 级 lineage 快照。
3. stage 04 zip 不包含 01、02、03 的 records、evaluation_report、aggregate_report 或完整上游 zip 内容。